# 07 — Toyota Smarthome extraction (OFFLINE, Blackwell)

| attach as input | produces |
|---|---|
| `behaviorsense-code`, `behavioursense-WW`, `behaviorsense-toyota-meta` | `behaviorsense-toyota-shards` |
| `toyota-smarthome-rgb` (MODE="trimmed") **or** `rgb-untrimmed` (MODE="untrimmed") | |
| **resume:** `behaviorsense-toyota-shards` (its own previous version) | |

**Internet OFF, Blackwell.** Runs RTMO over Toyota Smarthome and writes skeleton windows in
the same shard layout as `behaviorsense-adl-shards`, so the trainer reads both without a new
loader.

## Why RTMO and not the skeletons that ship with the dataset

Toyota provides 3D skeletons (V1.1 from LCR-Net, V1.2 refined by SSTA-PRS). They are not used
as the model's input, and that is deliberate: they are **15-joint 3D**, and the deployment
estimator is **RTMO producing 17-joint COCO 2D**. Training on one and serving the other is a
guaranteed train/serve skew, and this project has already paid twice for that class of bug
(`normalise()` erasing falls; the fall head's val AUROC 1.000 against held-out 0.47). The
V1.2 skeletons are worth having as an occlusion-robust QA reference, which is a different job.

## The two modes, and why trimmed comes first

| | clips | frames to pose | est. GPU | what it buys |
|---|---|---|---|---|
| `trimmed` | 16,115 | ~0.8 M at stride 2 | **~2 h** | clip-level labels for the window classifier |
| `untrimmed` | 536 videos, 149 h | ~6.7 M | **~9 h** | dense per-frame labels; f-mAP against PDAN's 32.7% |

Trimmed is what one session buys and it is what Agent 2 actually is: a window classifier.
Untrimmed is four times the cost and its value is mostly *evaluation*, so
`UNTRIMMED_SIDE = "test"` extracts only the 185 CS test videos (~3.5 h) and leaves the 351
training videos for a later session. Both modes write to the same dataset and resume from it.

**Set `MODE` in the first code cell.** Everything else is resolved by content.

In [ ]:
# Configuration. The only cell to edit.
MODE = "trimmed"           # "trimmed" (~7 h) or "untrimmed" (~3.5 h for the CS test side)
UNTRIMMED_SIDE = "test"    # CS side when MODE="untrimmed": test | train | both
FPS_SAMPLE = 20.0          # see below - measured, not chosen
MAX_CLIPS = 99999          # cap for a short session; shards already flushed are kept
WINDOWS_PER_SHARD = 8000

# FPS_SAMPLE = 20, and the first full run is why. It ran at 12.5, and because every trimmed
# container reports exactly 20 fps (`source frame rates seen: {20: 16115}`), the integer
# stride came out `round(20/12.5) = 2` - an effective 10 Hz. At 10 Hz a 30-frame window
# spans 3.0 s, so any clip shorter than 3 s produced NO window at all:
#
#   6,040 of 16,115 clips unusable (37.5%), and the loss was not uniform. `Sitdown` kept
#   233 windows and `Getup` 149, against annotated corpus shares of 0.72% and 0.67% -
#   three to four times under-represented, because a sit-to-stand IS a short clip. Those
#   two classes are what Agent 3 counts `sit_to_stand_count` from, and that is a validated
#   clinical frailty marker, so starving them undermines a headline feature rather than a
#   tail statistic.
#
# At 20 Hz the stride is 1, the window spans 1.50 s, and a clip needs only 1.5 s to yield
# one. Estimated ~322k windows against 97,787 - and 1.50 s is CLOSER to the Charades
# shards' 2.0 s than 3.00 s was, so window-extent consistency improves at the same time.
#
# It costs about twice the GPU: the first run took 3.55 h, so budget ~7 h of a 12 h session.
# `MAX_CLIPS` plus the resume path exist for the case where that runs out.
#
# The rate is part of the shard name (see PREFIX), so windows sampled at different rates can
# never end up in one training set by accident.
import subprocess, sys, pathlib
INPUT = pathlib.Path("/kaggle/input")

_MARKERS = {"torch", "rtmlib", "onnxruntime-gpu", "nvidia-cudnn-cu12", "triton"}
_dirs = {}
for _shape in ("datasets/*/*/wheels/*.whl", "datasets/*/*/*/wheels/*.whl",
               "datasets/*/*/*.whl"):
    for _w in INPUT.glob(_shape):
        _dirs.setdefault(_w.parent, set()).add(
            _w.name.split("-")[0].lower().replace("_", "-"))
    if _dirs:
        break
assert _dirs, "no wheels attached - behavioursense-WW carries them"
WHEEL_DIR, _hits = max(_dirs.items(), key=lambda kv: len(kv[1] & _MARKERS))
assert _hits & _MARKERS, f"wheel dir holds none of {sorted(_MARKERS)}: {WHEEL_DIR}"

# rtmlib pulls the CPU onnxruntime; both own the same module directory, so the CPU build
# silently clobbers the GPU one's provider registration. Remove it, install rtmlib without
# deps, put onnxruntime-gpu last. Same order as notebook 01, for the same measured reason.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                "--find-links", str(WHEEL_DIR), "rtmlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                "--find-links", str(WHEEL_DIR),
                "onnxruntime-gpu", "opencv-python-headless", "PyYAML"], check=True)
print(f"MODE={MODE}  FPS_SAMPLE={FPS_SAMPLE}  installed from {WHEEL_DIR}")

In [ ]:
# Resolve every attached asset by CONTENT, not by dataset name.
#
# Kaggle mount paths are not predictable from here, and three separate things vary:
#   - notebook 00 emits two folders, which can be published as ONE dataset or two
#     (observed: a single "behavioursense-WW" holding both wheels/ and weights/)
#   - the dataset title is free text, and "behaviour" vs "behavior" both occur
#   - Save Version nests the working directory inside the dataset, so files end up at
#     <mount>/kaggle/working/... rather than <mount>/...
#
# Guessing the name has already cost one session, and notebook 01's carry-forward bug
# showed how the failure presents: a wrong path reads as "nothing attached", the run
# continues, and work is skipped or destroyed rather than failing loudly.
#
# So identify each asset by a file only it has. A directory holding *.whl is the wheel
# cache no matter what the dataset is called.
import pathlib
from itertools import islice

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    # islice, NOT sorted(...)[:6]. `sorted()` materialises the whole listing
                    # first, and on Kaggle's FUSE mount a slug whose files sit at its root -
                    # `toyota-smarthome-skeleton-v1-2` holds 16,115 - makes that a full
                    # network directory read per mount. Eleven mounts of that shape is most
                    # of the 14 minutes this cell took on the first Toyota run. Six names
                    # are all this diagnostic needs, so stop after six.
                    top = sorted(islice((q.name for q in ds.iterdir()), 6)) \
                        if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []

# MOUNT-INDEXED SEARCH, and why two earlier fixes were not enough.
#
# `INPUT.glob("**/x")` walks every directory under /kaggle/input - 93 minutes with the
# Toyota corpus mounted, notebook 06 measured, and 349 s for the single
# `**/src/behaviorsense/__init__.py` probe in notebook 07's second run. The first "fix",
# FIXED-DEPTH globs like `datasets/*/*/*/rtmo-l.onnx`, was depth-bounded but not
# COST-bounded: to match at depth 3 pathlib scandirs EVERY slug child directory, including
# `toyota-smarthome-skeleton-v1-2`'s 16,115-file root and MSMT17's 65,242 crops.
#
# The mounts are KNOWN at depth 2 (`datasets/<owner>/<slug>/`), so enumerate them once and
# resolve everything else with is_file() stats - one metadata call per mount per candidate,
# never a sibling-directory listing.
_SLUGS = (sorted((INPUT / "datasets").glob("*/*"))
          + sorted((INPUT / "competitions").glob("*")))

def find_fast(tail, what, required=True):
    # Known staging prefixes, each costing one stat per mount:
    #   ""                      files at the slug root
    #   EmotionSense-Extended/  the code dataset was created by zipping the repo FOLDER,
    #                           so everything sits one level below the slug
    #   kaggle/working/         Save Version nests the working directory
    # The prefixes apply to MULTI-COMPONENT tails too. They used to be tried only for bare
    # filenames, which quietly sent `src/behaviorsense/__init__.py` - the one probe every
    # notebook makes - down the deep-search path it was written to avoid.
    # `weights/` is additionally tried for a bare filename, the staged weights layout.
    prefixes = ("", "EmotionSense-Extended/", "kaggle/working/")
    mids = ("",) if "/" in tail else ("", "weights/")
    for t in [p + m + tail for p in prefixes for m in mids]:
        hits = [s / t for s in _SLUGS if (s / t).is_file()]
        if hits:
            return hits[0]
    hits = sorted(INPUT.glob(f"**/{tail}"))   # last resort: unusual layout, slow, once
    if hits:
        print(f"  {what:<9} found only by deep search ({tail}) - layout is unusual")
        return hits[0]
    if required:
        raise AssertionError(
            f"{what}: nothing matches {tail!r} in any mount. Attached: {ATTACHED}")
    print(f"  {what:<9} ABSENT (optional)")
    return None

def find_asset(pattern, what, required=True):
    # Callers pass a `**/...` pattern. The leading `**/` is stripped and the mount-indexed
    # search runs first, so every existing call site gets the speed-up unchanged.
    tail = pattern[3:] if pattern.startswith("**/") else pattern
    return find_fast(tail, what, required=required)

def find_dir(subdir, pattern, roots=None):
    # "Which mount holds the most files matching this pattern in this subdirectory?" - one
    # scandir of ONE named directory per mount, never a recursive walk. Used for corpora
    # (Toyota's mp4/ and Videos_mp4/) where the answer is a directory, not a file.
    from fnmatch import fnmatch
    import os
    best, best_n = None, 0
    for root in (roots if roots is not None else _SLUGS):
        base = root / subdir if subdir else root
        if not base.is_dir():
            continue
        n = sum(1 for e in os.scandir(base) if e.is_file() and fnmatch(e.name, pattern))
        if n > best_n:
            best, best_n = base, n
    return best, best_n

def find_charades_csv():
    # The charades-480p dataset nests the CSV one level down (its root holds
    # Charades_annotations/ and Charades_v1_480/), and a `**` glob for it walks the code
    # dataset's MSMT17 copy - minutes for one file. Probe the known shapes per mount;
    # only a genuinely unknown layout falls through to the deep search.
    for slug in _SLUGS:
        if "charades" not in slug.name.lower():
            continue
        for base in (slug, slug / "Charades_annotations", slug / "Charades_v1_480",
                     slug / "kaggle" / "working"):
            p = base / "Charades_v1_train.csv"
            if p.is_file():
                return [p]
    return sorted(INPUT.glob("**/Charades_v1_train.csv"))

def find_wheel_dir():
    # "The directory containing *.whl" is not specific enough: /kaggle/input also holds
    # attached COMPETITIONS, and at least one (arc-prize-2026) ships its own wheels. The
    # first sorted hit was that competition's, and the offline install then failed on a
    # cache that simply does not contain torch. Score candidate directories by how many
    # of OUR packages they hold and take the best.
    MARKERS = {"torch", "rtmlib", "onnxruntime-gpu", "nvidia-cudnn-cu12", "triton"}
    dirs = {}
    for slug in _SLUGS:
        for wdir in (slug / "wheels", slug / "kaggle" / "working" / "wheels"):
            if not wdir.is_dir():
                continue
            for w in wdir.glob("*.whl"):
                dirs.setdefault(wdir, set()).add(
                    w.name.split("-")[0].lower().replace("_", "-"))
        if dirs:
            break
    if not dirs:
        for w in INPUT.glob("**/*.whl"):        # last resort
            dirs.setdefault(w.parent, set()).add(
                w.name.split("-")[0].lower().replace("_", "-"))
    if not dirs:
        raise AssertionError(f"no *.whl anywhere under /kaggle/input. Attached: {ATTACHED}")
    best, hits = max(dirs.items(), key=lambda kv: len(kv[1] & MARKERS))
    if not (hits & MARKERS):
        raise AssertionError(
            f"found {len(dirs)} wheel director(ies) but none holds any of {sorted(MARKERS)} "
            f"- the staged cache from notebook 00 is not attached. Candidates: "
            f"{[str(d) for d in dirs]}")
    return best

WHEELS  = find_wheel_dir()
WEIGHTS = find_asset("**/rtmo-l.onnx", "weights").parent
SRC     = find_asset("**/src/behaviorsense/__init__.py", "code").parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"

CONFIGS = CODE / "configs"

# sys.path belongs HERE, in the cell that resolves SRC, and unconditionally.
#
# It used to live at the end of the wheel-install cell. Notebook 04 put it outside that
# cell's `if not sm120_ok()` branch and worked; notebook 03 never had it at all and worked
# anyway, because every heavy step there is a subprocess launched with PYTHONPATH set. Then
# `is_real_artifact` was added to notebook 03's shard resolver - the first in-process import
# of `behaviorsense` in that notebook - and the next run died at cell 4 with
# `ModuleNotFoundError: No module named 'behaviorsense'`, six minutes in, one cell after
# PREFLIGHT PASSED. A path set up as a side effect of an unrelated, conditional cell is a
# dependency nobody can see.
import sys
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(SRC))

for _label, _path in (("wheels", WHEELS), ("weights", WEIGHTS), ("code", CODE)):
    print(f"  {_label:<8} {_path}")
print(f"  attached  {ATTACHED}")
print("NOTE: if you restart the kernel below, re-run from THIS cell - these names "
      "are what every later cell uses.")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_adl.py", "--tau-train",
     "notebook 03 passes --sampler/--tau-train; a snapshot predating them exits 2 from "
     "argparse, which notebook 03 reports as a failed stream rather than stale code"),
    ("src/behaviorsense/data/skeleton_dataset.py", "def load_subject_map",
     "video-id -> Charades actor-id remap for a person-disjoint P1 split (notebooks "
     "03/04). A stale snapshot silently reverts P1 to video-disjoint - same person in "
     "train and val - while printing numbers that look identical"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/ensemble.py", "def flip_windows",
     "test-time flip augmentation (notebook 04 levers cell). A stale snapshot would accept "
     "`clf.tta = True` as a new attribute and silently do no TTA, reporting the "
     "unaugmented number as if it were augmented"),
    ("src/behaviorsense/models/stgcnpp.py", "parents.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise. The "
     "token tracked the local name `parent` and broke when the function grew an explicit "
     "`parents` argument for Toyota's 15-node tree - a rename silently disarming a staleness "
     "guard is exactly what this list exists to catch, so it caught itself"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def force_greedy",
     "pins BOTH decoding arms to greedy (notebook 04). Without it the constrained arm "
     "inherits Qwen's generation_config (do_sample=True, temperature=0.7) while the free "
     "arm is greedy, so the comparison measures temperature instead of grammar - the "
     "constrained rate moved 8.2% -> 10.3% between two runs of identical code"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def _chat_text",
     "both arms send the SAME templated text (notebook 04). The constrained path used to "
     "hand outlines the raw prompt, so one arm got a Qwen chat turn and the other a naked "
     "instruction block - a second confound on top of the sampling one"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/eval/activity_eval.py", "def logit_adjust",
     "notebook 04's P1 cell imports MIN_SUPPORT and scores() from here, so a stale "
     "snapshot fails with ImportError at cell 3; also carries the post-hoc accuracy "
     "levers scripts/rescore_p1.py replays off the saved val logits"),
    ("src/behaviorsense/video.py", "def child_env",
     "notebook 05's /video endpoint decodes uploads in a CHILD process (ffmpeg raises SIGSEGV "
     "on malformed streams and a signal is not catchable, so without the boundary one bad "
     "upload kills the kernel, the tunnel and the demo together). `child_env` is what puts "
     "behaviorsense on that child's PYTHONPATH - sys.path does not cross a process boundary, "
     "and a snapshot without it 422s EVERY upload with \"No module named 'behaviorsense'\""),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
# Prove the GPU is in use BEFORE spending the session. onnxruntime falls back to CPU
# silently - correct poses at 1/50th the speed - and that presents as "the GPU is slow"
# rather than "the GPU is not being used". Notebook 01 lost a session to exactly this.
import onnxruntime as ort
providers = ort.get_available_providers()
print("onnxruntime providers:", providers)
assert any("CUDA" in p for p in providers), (
    f"no CUDAExecutionProvider: {providers}. The staged onnxruntime-gpu did not load - "
    "check the CPU build was removed before it was installed.")

import collections, cv2, json, os, shutil, time
import numpy as np

# `find_dir` and `find_asset` come from the resolver cell above (MOUNT_PRELUDE), which is
# the same code notebooks 01-06 use. This cell used to define a second, local `find_dir`
# over its own `ROOTS` while calling `find_asset` from a cell that did not export it -
# NameError after 370 s of session. One definition, one place.
ROOTS = _SLUGS

# The meta files sit at <slug>/toyota_meta/, NOT the slug root: notebook 06 wrote them to
# /kaggle/working/toyota_meta/ and Save Version nests the working directory inside the
# dataset. The first run of this notebook failed here for exactly that reason - the mount
# was attached, the assert just looked one level too shallow. Try the known layout first,
# then the root, and name what was actually seen before giving up.
META, _ = find_dir("toyota_meta", "smarthome_CS_51.json")
if META is None:
    META, _ = find_dir("", "smarthome_CS_51.json")
assert META is not None, (
    "attach behaviorsense-toyota-meta (notebook 06's output). This notebook is OFFLINE "
    "and cannot fetch the annotations. Looked for <mount>/toyota_meta/smarthome_CS_51.json "
    f"and <mount>/smarthome_CS_51.json. Mounts: {[r.name for r in ROOTS]}")
if MODE == "trimmed":
    VIDEO_DIR, N_VID = find_dir("mp4", "*_p[0-9][0-9]_r*_c[0-9][0-9].mp4")
else:
    VIDEO_DIR, N_VID = find_dir("Videos_mp4", "P*T*C*.mp4")
assert VIDEO_DIR is not None, f"no {MODE} RGB mount. Mounts: {[r.name for r in ROOTS]}"
RTMO_PATH = str(WEIGHTS / "rtmo-l.onnx")
# WEIGHTS comes from the RESOLVE cell above, which already located the weights mount by
# content. Re-resolving here with `find_asset` was how version 2 of this notebook died: it
# used the lighter RESOLVE_CODE cell, which resolves only the repo, so `find_asset` was
# never exported - NameError 370 s into the session, on a line unrelated to the real cause.
assert pathlib.Path(RTMO_PATH).is_file(), f"no rtmo-l.onnx under {WEIGHTS}"
print(f"meta   {META}")
print(f"video  {VIDEO_DIR}  ({N_VID:,} files)")
print(f"rtmo   {RTMO_PATH}")

In [ ]:
# The work list, and resume. Shards already written are kept; a session that times out is
# resumed by attaching this notebook's own previous output version, exactly as notebook 01
# carries `behaviorsense-adl-shards` forward.
from behaviorsense.data.toyota import (CS_TEST_SUBJECTS, CS_TRAIN_SUBJECTS, COARSE_V11,
                                       TSM_CLASSES, TSM_TO_COARSE, TSU_CLASSES,
                                       TSU_TO_COARSE, collapse_matrix, frame_mask,
                                       frame_multilabel, load_tsu_annotations,
                                       parse_tsm_name, parse_tsu_filename, protocol_side)

OUT = pathlib.Path("/kaggle/working/shards"); OUT.mkdir(parents=True, exist_ok=True)
# The sampling rate is in the shard name. Without it, re-running at a different FPS_SAMPLE
# would resume against shards whose windows span a different number of seconds, and the two
# rates would mix silently inside one training set - each clip contributing twice, at two
# different speeds, with no field recording which. A rate change now forces a clean
# extraction instead.
PREFIX = f"toyota_{MODE}_{FPS_SAMPLE:g}hz"

DONE = set()
# Carry forward EVERY toyota shard, not just this run's prefix, and this is a data-loss
# guard rather than tidiness. Save Version publishes whatever sits in /kaggle/working: an
# untrimmed run that copied only `toyota_untrimmed_*` forward would produce a new version of
# `behaviorsense-toyota-shards` containing ONLY the untrimmed half, silently dropping the
# 214,913 trimmed windows from the latest version. Copy everything in, add to it, publish the
# union.
#
# Mount-indexed, like every other lookup: stat <slug>/shards/ per mount instead of globbing.
# Even a FIXED-DEPTH glob (`datasets/*/*/*/shards/...`) makes pathlib scandir every slug
# child directory - the 16k skeleton root and MSMT17's crops included - which is the ~8.5
# minutes the first run of this notebook spent in its resolver cell. is_dir() guards each
# candidate, so nothing large is ever listed.
_prev_hits: list = []
for _slug in ROOTS:
    for _cand in (_slug / "shards", _slug / "kaggle" / "working" / "shards"):
        if _cand.is_dir():
            _prev_hits += list(_cand.glob("toyota_*.npz"))
for _prev in _prev_hits:
    if not (OUT / _prev.name).exists():
        shutil.copy2(_prev, OUT / _prev.name)
# DONE is built from THIS prefix only. Clip names do not collide between the halves
# (`Walk_p03_r01_v15_c07.mp4` against `P15T17C03.mp4`), but keying resume on the rate and
# mode that produced a shard is what makes a future rate change safe.
for _sh in sorted(OUT.glob(f"{PREFIX}_*.npz")):
    with np.load(_sh, allow_pickle=False) as _z:
        if "clips" in _z.files:
            DONE.update(str(c) for c in _z["clips"])
_carried = sorted(p.name.rsplit("_", 1)[0] for p in OUT.glob("toyota_*.npz"))
print(f"carried forward {len(list(OUT.glob('toyota_*.npz')))} shard(s) across "
      f"{len(set(_carried))} shard set(s): {sorted(set(_carried))}")
print(f"resuming {PREFIX}: {len(list(OUT.glob(PREFIX + '_*.npz')))} shard(s), "
      f"{len(DONE):,} clip(s) already extracted")

ANN, M_FINE = None, None
if MODE == "trimmed":
    WORK, skipped = [], 0
    for p in sorted(VIDEO_DIR.glob("*.mp4")):
        try:
            v = parse_tsm_name(p.name)
        except ValueError:
            skipped += 1                   # not a trimmed clip name; skip, never guess
            continue
        if v.activity not in TSM_TO_COARSE:
            skipped += 1                   # outside the 31-class space, counted not hidden
            continue
        WORK.append((p, v))
    print(f"{len(WORK):,} clips in the 31-class space, {skipped:,} outside it or unparseable")
else:
    ANN = load_tsu_annotations(META / "smarthome_CS_51.json")
    M_FINE = collapse_matrix(TSU_CLASSES, TSU_TO_COARSE)
    sides = ("train", "test") if UNTRIMMED_SIDE == "both" else (UNTRIMMED_SIDE,)
    WORK = []
    for p in sorted(VIDEO_DIR.glob("P*T*C*.mp4")):
        v = parse_tsu_filename(p)
        if protocol_side(v, "CS") in sides and v.tsu in ANN:
            WORK.append((p, v))
    print(f"{len(WORK)} untrimmed videos on the CS {'+'.join(sides)} side")

WORK = [w for w in WORK if w[0].name not in DONE][:MAX_CLIPS]
print(f"{len(WORK):,} to do this session")

In [ ]:
# Extract. RTMO per frame, slot assignment, fixed windows - all through the repo's own
# helpers so this notebook cannot drift from what the trained model expects.
from rtmlib import RTMO
from prepare_skeletons import WINDOW_FRAMES, assign_slots, window_clip

body = RTMO(onnx_model=RTMO_PATH, model_input_size=(640, 640),
            backend="onnxruntime", device="cuda")
_sess = getattr(body, "session", None)
assert _sess is None or any("CUDA" in p for p in _sess.get_providers()), (
    f"RTMO loaded on {_sess.get_providers()}, not CUDA")

def poses_for(path, fps_sample=FPS_SAMPLE, cap_frames=40_000):
    # -> ([T,M,17,3], source fps, effective rate, reason-if-empty). The rate is READ rather
    # than assumed: notebook 06 measured the trimmed containers at 20 fps while their own
    # README documents 30, and the first full run confirmed it on all 16,115 clips.
    #
    # EXACT resampling, not integer striding. Integer striding cost the first run 6,040 clips
    # (37.5%): at 20 fps a nominal 12.5 Hz rounds to stride 2, an effective 10 Hz, a 3.0 s
    # window, and every clip under 3 s produced nothing - concentrated in exactly the short
    # transition classes Agent 3 counts sit_to_stand_count from.
    #
    # It also cannot hold the window duration constant across the two halves: 25 fps
    # untrimmed strides to 25 Hz (1.20 s) while 20 fps trimmed strides to 20 Hz (1.50 s), so
    # the same 30-frame window would span different real time in the two shard sets. Picking
    # frame k as round(k * src / fps_sample) hits the requested rate on both, so a window is
    # 1.50 s everywhere and the two halves are directly comparable.
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None, 0.0, 0.0, "decoder refused the file"
    src = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    if not (1.0 <= src <= 240.0):
        src = 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    rate = min(fps_sample, src)          # never invent frames the container does not have
    if total and int(total * rate / src) < WINDOW_FRAMES:
        cap.release()
        return None, src, rate, f"only {total} frames at {src:g} fps, needs {WINDOW_FRAMES}"
    out, i, k = [], 0, 0
    want = 0
    while len(out) < cap_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if i == want:
            kp, sc = body(frame)
            out.append(assign_slots(np.asarray(kp), np.asarray(sc), out[-1] if out else None))
            k += 1
            want = int(round(k * src / rate))
        i += 1
    cap.release()
    if len(out) < WINDOW_FRAMES:
        return None, src, rate, f"decoded {len(out)} frames, needs {WINDOW_FRAMES}"
    return np.stack(out), src, rate, ""

skels, fine, coarse, subs, srcs, clips, starts = [], [], [], [], [], [], []
shard_idx = len(list(OUT.glob(PREFIX + "_*.npz")))
n_ok = n_empty = n_gap = 0
src_rates = collections.Counter()
# Named drop reasons and the strides actually used - not one opaque "unusable" count. The
# first run reported 6,040 unusable and nothing about WHY, so finding the striding cause
# needed the class histogram plus arithmetic afterwards rather than the log itself.
drop_reasons = collections.Counter()
rates = collections.Counter()
t0 = time.time()

def flush():
    global skels, fine, coarse, subs, srcs, clips, starts, shard_idx
    if not skels:
        return
    np.savez_compressed(
        OUT / f"{PREFIX}_{shard_idx:04d}.npz",
        skeletons=np.stack(skels).astype(np.float16),
        # `labels` is the COARSE id so the existing loader's shape is unchanged;
        # `fine_labels` carries the 31/51-class id the benchmark head trains on. Both are
        # written because collapsing before the loss destroys the supervision that separates
        # the members - all five Cook.* classes become one target - and collapsing after
        # keeps every discriminative gradient.
        labels=np.asarray(coarse, dtype=np.int64),
        fine_labels=np.asarray(fine, dtype=np.int64),
        subjects=np.asarray(subs, dtype="<U32"),
        datasets=np.asarray(srcs, dtype="<U32"),
        clips=np.asarray(clips, dtype="<U64"),
        # The window's start index WITHIN its clip, at the sampled rate. Without it a
        # shard is an unordered bag: the kept windows are not an arithmetic sequence
        # (the visibility gate and the gap mask both drop some), so neither the temporal
        # order for segment metrics nor a per-frame timeline for mAP can be rebuilt.
        # Eight bytes a window against a seven-hour re-extraction.
        starts=np.asarray(starts, dtype=np.int64))
    print(f"  shard {shard_idx}: {len(skels):,} windows")
    shard_idx += 1
    skels, fine, coarse, subs, srcs, clips, starts = [], [], [], [], [], [], []

for seen, (path, vid) in enumerate(WORK, start=1):
    poses, src_fps, eff_rate, why = poses_for(path)
    src_rates[round(src_fps)] += 1
    if eff_rate:
        rates[round(eff_rate, 1)] += 1
    if poses is None:
        n_empty += 1
        drop_reasons[why or "unknown"] += 1
    else:
        windows = window_clip(poses, with_starts=True)
        if not windows:
            n_empty += 1
            drop_reasons["no window passed the visibility gate"] += 1
        elif MODE == "trimmed":
            f_id = TSM_CLASSES.index(vid.activity)
            c_id = COARSE_V11.index(TSM_TO_COARSE[vid.activity])
            for _start, w in windows:
                skels.append(w); fine.append(f_id); coarse.append(c_id)
                subs.append(f"p{vid.subject:02d}"); srcs.append("toyota_trimmed")
                clips.append(path.name); starts.append(int(_start))
            n_ok += 1
        else:
            # Dense labels. The multilabel tensor is built at the SAMPLED length so a
            # window's frame range maps back through the same step the decoder used.
            v = ANN[vid.tsu]
            n_steps = len(poses)
            lab = frame_multilabel(v, n_steps, official=False)
            msk = frame_mask(v, n_steps)
            added = 0
            for start, w in windows:
                sl = slice(start, start + WINDOW_FRAMES)
                if msk[sl].mean() < 0.5:
                    n_gap += 1
                    continue               # mostly unannotated: no supervised class here
                counts = lab[sl].sum(axis=0)
                f_id = int(counts.argmax())
                if counts[f_id] < WINDOW_FRAMES * 0.5:
                    n_gap += 1
                    continue               # no single class covers half the window
                skels.append(w); fine.append(f_id)
                coarse.append(int(np.argmax(M_FINE[f_id])))
                subs.append(f"p{vid.subject:02d}"); srcs.append("toyota_untrimmed")
                clips.append(path.name); starts.append(int(start))
                added += 1
            n_ok += 1 if added else 0
            n_empty += 0 if added else 1
    if len(skels) >= WINDOWS_PER_SHARD:
        flush()
    if seen % 500 == 0 or seen == len(WORK):
        el = time.time() - t0
        eta = el / seen * (len(WORK) - seen) / 60
        print(f"  {seen:,}/{len(WORK):,} clips, {n_ok:,} ok, {n_empty:,} unusable, "
              f"{len(skels):,} pending, {el / 60:.1f} min, eta {eta:.0f} min")
flush()
print(f"\n{n_ok:,} clips extracted, {n_empty:,} unusable, {n_gap:,} windows dropped as gap")
print(f"source frame rates seen: {dict(src_rates)}")
print(f"effective sample rates: {dict(rates)} Hz   (exact resampling, so a 30-frame "
      f"window is {30 / FPS_SAMPLE:.2f}s on BOTH halves regardless of container rate)")
if drop_reasons:
    print("why clips were unusable:")
    for _r, _n in drop_reasons.most_common():
        print(f"  {_n:>6,}  {_r}")
print(f"{time.time() - t0:.0f}s total")

In [ ]:
# Verify what was written BEFORE Save Version. A shard set that trains but is mislabelled is
# the expensive failure, so every check here is on the CONTENT rather than the file count.
shards = sorted(OUT.glob(PREFIX + "_*.npz"))
assert shards, "no shards written - do NOT Save Version"

N, fine_hist, coarse_hist, subj = 0, collections.Counter(), collections.Counter(), set()
for sh in shards:
    with np.load(sh, allow_pickle=False) as z:
        N += len(z["labels"])
        fine_hist.update(z["fine_labels"].tolist())
        coarse_hist.update(z["labels"].tolist())
        subj.update(z["subjects"].tolist())
        assert z["skeletons"].shape[1:] == (WINDOW_FRAMES, 2, 17, 3), z["skeletons"].shape
        assert z["skeletons"].dtype == np.float16, z["skeletons"].dtype
        assert len(z["labels"]) == len(z["fine_labels"]) == len(z["subjects"])
        # `starts` is what makes a shard an ORDERED record rather than a bag of windows.
        # Older shards predate it; say so instead of failing, because the trimmed half's
        # clip-level labels need no timeline and are still usable without it.
        if "starts" not in z.files:
            print(f"  NOTE {sh.name} has no `starts` - segment metrics and per-frame mAP "
                  "cannot be rebuilt from it")
        else:
            assert len(z["starts"]) == len(z["labels"])

FINE_NAMES = TSM_CLASSES if MODE == "trimmed" else TSU_CLASSES
print(f"{N:,} windows in {len(shards)} shard(s), {len(subj)} subjects: {sorted(subj)}")
assert max(fine_hist) < len(FINE_NAMES), f"fine label {max(fine_hist)} outside the id space"
assert max(coarse_hist) < len(COARSE_V11), f"coarse label {max(coarse_hist)} out of range"

# The CS partition must be intact IN THE SHARDS, not just in the plan.
_tr = {s for s in subj if int(s[1:]) in CS_TRAIN_SUBJECTS}
_te = {s for s in subj if int(s[1:]) in CS_TEST_SUBJECTS}
print(f"CS subjects present - train {len(_tr)}, test {len(_te)}")
assert not (_tr & _te), "a subject appears on both CS sides"

print(f"\n{'fine class':<34}{'windows':>9}{'share%':>8}")
for i, n in fine_hist.most_common(15):
    print(f"{FINE_NAMES[i]:<34}{n:>9,}{100 * n / N:>7.2f}")
if len(fine_hist) > 15:
    print(f"  ... and {len(fine_hist) - 15} more of {len(FINE_NAMES)} fine classes")
print(f"\n{'coarse class':<34}{'windows':>9}{'share%':>8}")
for i, n in coarse_hist.most_common():
    print(f"{COARSE_V11[i]:<34}{n:>9,}{100 * n / N:>7.2f}")
_idle = coarse_hist.get(COARSE_V11.index("other_idle"), 0)
print(f"\nother_idle: {_idle} windows. Zero is intended - every Toyota class is a real "
      "activity, which is the opposite of the Charades map where the fallback was largest.")
print(f"imbalance {max(fine_hist.values()) / max(1, min(fine_hist.values())):.0f}x over "
      f"{len(fine_hist)}/{len(FINE_NAMES)} fine classes present")

# MIN_SUPPORT mirrors the eval path: below ~50 windows a class's F1 is one prediction wide,
# so it is reported but not averaged. Naming those classes here is what stops a per-class
# mean being quoted over classes that cannot support one - the exact trap the Charades run
# fell into with `bending_reaching` at 26 windows.
MIN_SUPPORT = 50
_thin = sorted(((n, FINE_NAMES[i]) for i, n in fine_hist.items() if n < MIN_SUPPORT))
if _thin:
    print(f"\n{len(_thin)} fine class(es) below MIN_SUPPORT={MIN_SUPPORT} - report them, "
          "do NOT average them:")
    for _n, _name in _thin:
        print(f"  {_n:>5}  {_name}")
else:
    print(f"every present fine class has >= {MIN_SUPPORT} windows")

# What the OUTPUT DATASET will contain, not just what this run produced. A Save Version
# publishes every shard in /kaggle/working, so this is the line that says whether both halves
# survived the carry-forward.
_all = sorted(OUT.glob("toyota_*.npz"))
_sets: dict = {}
for _sh in _all:
    _key = _sh.name.rsplit("_", 1)[0]
    with np.load(_sh, allow_pickle=False) as _z:
        _sets[_key] = _sets.get(_key, 0) + len(_z["labels"])
print(f"\nthe dataset this Save Version will publish:")
for _key, _n in sorted(_sets.items()):
    print(f"  {_key:<28}{_n:>10,} windows")
print(f"  {'TOTAL':<28}{sum(_sets.values()):>10,} windows in {len(_all)} shard(s)")

pathlib.Path("/kaggle/working/toyota_shard_manifest.json").write_text(json.dumps({
    "mode": MODE, "fps_sample": FPS_SAMPLE, "window_frames": WINDOW_FRAMES,
    "untrimmed_side": UNTRIMMED_SIDE if MODE == "untrimmed" else None,
    "n_windows": N, "n_shards": len(shards), "subjects": sorted(subj),
    "source_frame_rates": {str(k): v for k, v in src_rates.items()},
    "effective_rates_hz": {str(k): v for k, v in rates.items()},
    "window_seconds": WINDOW_FRAMES / FPS_SAMPLE,
    "drop_reasons": dict(drop_reasons),
    "fine_histogram": {FINE_NAMES[i]: n for i, n in fine_hist.items()},
    "coarse_histogram": {COARSE_V11[i]: n for i, n in coarse_hist.items()},
    "clips_extracted": n_ok, "clips_unusable": n_empty, "windows_dropped_as_gap": n_gap,
}, indent=1), encoding="utf-8")
print("wrote /kaggle/working/toyota_shard_manifest.json")

print()
print("=" * 70)
print("  Save Version -> create/update PRIVATE dataset behaviorsense-toyota-shards")
print("=" * 70)
if MODE == "untrimmed" and UNTRIMMED_SIDE != "both":
    print(f"This run covered the CS {UNTRIMMED_SIDE} side only. Attach this dataset back as")
    print("an input, switch UNTRIMMED_SIDE, and re-run: carried-forward shards are kept and")
    print("their clips are skipped.")
elif len(WORK) >= MAX_CLIPS:
    print("MAX_CLIPS was hit, so this run did not cover everything. Attach this dataset")
    print("back as an input and re-run to continue where it stopped.")